# ArcFace subject-consistency test (face-crop vs face-crop)
Tests whether ArcFace identity similarity tracks GPT-4.1 SC where global DINO/CLIP failed. Broad (all ref-with-face rows) + strict (unambiguous single-face rows).

## 1. Setup — Drive, data, insightface

In [ ]:
# mount Drive, data, insightface (buffalo_l = RetinaFace detect + ArcFace recog)
from google.colab import drive
import os, subprocess, sys
drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "insightface", "onnxruntime-gpu"], check=False)

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
GEN   = f"{DRIVE}/gen/baseline"
IMGS, EXPECTED = "/content/imgs", 37063
TAR   = f"{DRIVE}/imgs_subset.tar"
def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0
if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
if not os.path.exists("/content/test.jsonl"):
    subprocess.run(["cp", f"{DRIVE}/test.jsonl", "/content/test.jsonl"])

import numpy as np
from insightface.app import FaceAnalysis
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))

def to_bgr(pil): return np.array(pil.convert("RGB"))[:, :, ::-1]
def largest(faces):
    return max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1])) if faces else None

def row_arcface(ref_imgs, gen_img):
    gen_faces = app.get(to_bgr(gen_img)); gen_embs = [f.normed_embedding for f in gen_faces]
    ref_faces = [app.get(to_bgr(r)) for r in ref_imgs]
    ref_counts = [len(fl) for fl in ref_faces]
    # BROAD: each ref with a face -> its largest face best-matched to any generated face, averaged
    sims = []
    for fl in ref_faces:
        lf = largest(fl)
        if lf is not None and gen_embs:
            sims.append(max(float(lf.normed_embedding @ ge) for ge in gen_embs))
    broad = sum(sims) / len(sims) if sims else None
    # STRICT: 1 ref, exactly 1 face in ref AND 1 face in generated (unambiguous, no matching)
    strict = None
    if len(ref_imgs) == 1 and ref_counts == [1] and len(gen_faces) == 1:
        strict = float(ref_faces[0][0].normed_embedding @ gen_embs[0])
    return broad, strict, ref_counts, len(gen_faces)

print("insightface ready")

Mounted at /content/drive
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 73684.88KB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
insightface 

## 2. Sanity — face counts + cosine on row 0

In [ ]:
# sanity: face counts + broad/strict cosine on row 0
import json
from PIL import Image
rows = [json.loads(l) for l in open("/content/test.jsonl", encoding="utf-8")]
r = rows[0]
refs = [Image.open(os.path.join(IMGS, p)).convert("RGB") for p in r["input_images"]]
gen  = Image.open(f"{GEN}/test_00000.png").convert("RGB")
broad, strict, ref_counts, gen_count = row_arcface(refs, gen)
print("n_refs:", len(refs), "| faces per reference:", ref_counts, "| faces in generated:", gen_count)
print("arcface_broad:", broad, "| arcface_strict:", strict)

n_refs: 1 | faces per reference: [1] | faces in generated: 1
arcface_broad: 0.32502317428588867 | arcface_strict: 0.32502317428588867


## 3. Full pass — broad + strict ArcFace (resumable)

In [ ]:
# full pass: broad + strict ArcFace per row; resumable
import json
from PIL import Image
AF = f"{DRIVE}/scores/baseline_arcface.jsonl"
os.makedirs(os.path.dirname(AF), exist_ok=True)
done = set()
if os.path.exists(AF):
    for l in open(AF, encoding="utf-8"):
        try:
            d = json.loads(l)
            if "error" not in d: done.add(d["i"])
        except: pass
todo = [i for i in range(len(rows)) if i not in done]
print(f"to process: {len(todo)} (done: {len(done)})")
with open(AF, "a", encoding="utf-8") as f:
    for n, i in enumerate(todo, 1):
        r = rows[i]; gp = f"{GEN}/test_{i:05d}.png"
        if not os.path.exists(gp):
            f.write(json.dumps({"i": i, "error": "missing_generated"}) + "\n"); f.flush(); continue
        try:
            refs = [Image.open(os.path.join(IMGS, p)).convert("RGB") for p in r["input_images"]]
            gen  = Image.open(gp).convert("RGB")
            broad, strict, ref_counts, gen_count = row_arcface(refs, gen)
        except Exception as e:
            f.write(json.dumps({"i": i, "error": type(e).__name__}) + "\n"); f.flush(); continue
        ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
        f.write(json.dumps({"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])),
                            "n_refs": len(r["input_images"]), "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
                            "ref_faces": ref_counts, "gen_faces": gen_count,
                            "arcface_broad": broad, "arcface_strict": strict}) + "\n")
        f.flush()
        if n % 50 == 0: print(f"  {n}/{len(todo)}")
print("done ->", AF)

to process: 488 (done: 0)
  50/488
  100/488
  150/488
  200/488
  250/488
  300/488
  350/488
  400/488
  450/488
done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/baseline_arcface.jsonl


## 4. Correlate with GPT-4.1 SC (coverage + Pearson/Spearman)

In [ ]:
# correlate broad + strict ArcFace with GPT-4.1 SC; report coverage
import json
from scipy.stats import pearsonr, spearmanr

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
AF = f"{DRIVE}/scores/baseline_arcface.jsonl"
BP = f"{DRIVE}/scores/test__baseline.jsonl"
af = {d["i"]: d for d in (json.loads(l) for l in open(AF, encoding="utf-8")) if "error" not in d}
gp = {d["i"]: d for d in (json.loads(l) for l in open(BP, encoding="utf-8")) if "error" not in d}

def mean(xs): return sum(xs) / len(xs)

print("rows processed:", len(af))
n_ref_face = sum(1 for d in af.values() if any(c > 0 for c in d["ref_faces"]))
print("rows whose reference has >=1 face:", n_ref_face, "/", len(af), "(ArcFace's natural domain)")
for metric in ["arcface_broad", "arcface_strict"]:
    pairs = [(af[i][metric], gp[i]["SC"]) for i in af if i in gp and af[i].get(metric) is not None]
    print("\n%s: coverage %d/%d usable rows" % (metric, len(pairs), len(af)))
    if len(pairs) >= 3:
        x = [a for a, _ in pairs]; y = [b for _, b in pairs]
        print("  vs GPT-4.1 SC: pearson=%.3f  spearman=%.3f" % (pearsonr(x, y)[0], spearmanr(x, y).statistic))
        print("  mean arcface_sim=%.3f | SC mean on these rows=%.2f" % (mean(x), mean(y)))
print("\n(contrast: DINO r=0.04, CLIP r=0.11 vs GPT-4.1 SC)")

rows processed: 488
rows whose reference has >=1 face: 412 / 488 (ArcFace's natural domain)

arcface_broad: coverage 376/488 usable rows
  vs GPT-4.1 SC: pearson=0.234  spearman=0.258
  mean arcface_sim=0.307 | SC mean on these rows=8.32

arcface_strict: coverage 106/488 usable rows
  vs GPT-4.1 SC: pearson=0.293  spearman=0.269
  mean arcface_sim=0.343 | SC mean on these rows=8.42

(contrast: DINO r=0.04, CLIP r=0.11 vs GPT-4.1 SC)
